<a href="https://colab.research.google.com/github/Sitthisak17SM/Super_AI/blob/main/Sleep_Stage_600817.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from google.colab import userdata

In [ ]:
# --- 1. Setup Kaggle & Download Data ---

KAGGLE_API_KEY = userdata.get('KAGGLE_API')
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_KEY
!kaggle competitions download -c super-ai-engineer-ss-6-sleep-stage-classification
!unzip -q super-ai-engineer-ss-6-sleep-stage-classification

# --- 2. Configuration & Paths ---
TRAIN_DIR = Path("train/train")
TEST_DIR  = Path("test_segment/test_segment")
SAMPLE_SUB = Path("sample_submission.csv")

In [ ]:
# --- 3. Data Loader Function ---
def load_and_prepare_data(train_dir: Path):
    all_epochs = []
    all_labels = []

    FEATURE_COLS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
    LABEL_COL    = "Sleep_Stage"
    EPOCH_LEN    = 480

    train_files = sorted(train_dir.glob("*.csv"))
    for csv_file in tqdm(train_files, desc="Loading CSVs"):
        df = pd.read_csv(csv_file)
        n_epochs = len(df) // EPOCH_LEN

        for i in range(n_epochs):
            epoch_data = df.iloc[i*EPOCH_LEN : (i+1)*EPOCH_LEN]
            signal = epoch_data[FEATURE_COLS].values
            label  = epoch_data[LABEL_COL].iloc[0]

            all_epochs.append(signal)
            all_labels.append(label)

    return all_epochs, all_labels

In [ ]:
# --- 4. Feature Extraction Function ---
def extract_features_v2(signal: np.ndarray) -> np.ndarray:
    features = []
    SR = 16

    # แยก Channel
    bvp, acc_x, acc_y, acc_z, temp, eda, hr, ibi = [signal[:, i] for i in range(8)]

    # Time & Freq Domain Features (แบบย่อแต่ครบถ้วน)
    for ch in range(signal.shape[1]):
        x = signal[:, ch]
        features += [np.mean(x), np.std(x), np.max(x)-np.min(x), pd.Series(x).skew()]

        # Frequency Power (Simple FFT)
        fft_vals = np.abs(np.fft.rfft(x))
        features.append(np.sum(fft_vals**2))

    # ACC Magnitude (Movement)
    acc_mag = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)
    features += [np.mean(acc_mag), np.std(acc_mag), np.max(acc_mag)]

    # HRV from IBI
    ibi_clean = ibi[ibi > 0]
    if len(ibi_clean) > 1:
        features += [np.std(ibi_clean), np.sqrt(np.mean(np.diff(ibi_clean)**2))]
    else:
        features += [0, 0]

    # Trends (Temp & EDA)
    features.append(np.polyfit(np.arange(len(temp)), temp, 1)[0]) # Temp slope
    features.append(np.mean(np.abs(np.diff(eda)))) # EDA Activity

    return np.array(features)

In [ ]:
# --- 5. Main Execution Flow ---
print("กำลังโหลดและเตรียมข้อมูล")
raw_epochs, labels = load_and_prepare_data(TRAIN_DIR)

print("Features")
X_train = np.array([extract_features_v2(ep) for ep in tqdm(raw_epochs)])

print("Label Encoding")
le = LabelEncoder()
y_train = le.fit_transform(labels)

In [ ]:
# --- 6. LightGBM Model & Pipeline ---
lgbm_params = {
    "n_estimators": 800,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "objective": "multiclass",
    "metric": "multi_logloss",
    "class_weight": "balanced", # Imbalance data
    "random_state": 42,
    "n_jobs": -1
}

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", lgb.LGBMClassifier(**lgbm_params))
])

# Cross-Validation
print("\n🔄 Running Cross-Validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_weighted')
print(f"✅ Mean Weighted F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Train Final Model
pipeline.fit(X_train, y_train)

In [ ]:
# --- 7. Prediction & Submission ---
def make_submission(test_dir, model, le, sample_sub_path):
    results = []
    test_files = sorted(test_dir.rglob("*.csv"))

    for f in tqdm(test_files, desc="Predicting"):
        df_test = pd.read_csv(f)
        feat = extract_features_v2(df_test[["BVP","ACC_X","ACC_Y","ACC_Z","TEMP","EDA","HR","IBI"]].values)
        pred = model.predict(feat.reshape(1, -1))
        label_str = le.inverse_transform(pred)[0]
        results.append({"id": f.stem, "labels": label_str})

    sub_df = pd.DataFrame(results)
    sample_df = pd.read_csv(sample_sub_path)
    # Merge เพื่อให้ลำดับ id ตรงกับไฟล์ส่งงาน
    final_sub = sample_df[['id']].merge(sub_df, on='id', how='left')
    final_sub.to_csv("submission_lgbm.csv", index=False)
    return final_sub

print("\n Submission")
final_df = make_submission(TEST_DIR, pipeline, le, SAMPLE_SUB)
print("Done")